A data-driven inventory and demand forecasting engine designed to bridge historical sales trends with warehouse logistics, empowering businesses to mitigate stockout risks and make smarter procurement decisions

In [1]:
import sqlite3
from datetime import datetime, timedelta
import random

def setup_ecommerce_database(db_name="ecommerce_demand.db"):
    # Connect to SQLite database (creates it if it doesn't exist)
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

    # 1. Create Products Table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Products (
            product_id INTEGER PRIMARY KEY AUTOINCREMENT,
            product_name TEXT,
            category TEXT,
            unit_price REAL,
            supplier_lead_time_days INTEGER
        )
    ''')

    # 2. Create Transactions Table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Transactions (
            transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
            date DATETIME,
            customer_id INTEGER,
            product_id INTEGER,
            quantity_sold INTEGER,
            total_revenue REAL,
            FOREIGN KEY (product_id) REFERENCES Products (product_id)
        )
    ''')

    # 3. Create Inventory Table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Inventory (
            inventory_id INTEGER PRIMARY KEY AUTOINCREMENT,
            product_id INTEGER,
            stock_on_hand INTEGER,
            safety_stock_threshold INTEGER,
            last_restock_date DATETIME,
            FOREIGN KEY (product_id) REFERENCES Products (product_id)
        )
    ''')

    # Insert Sample Products
    products = [
        ("Wireless Mouse", "Electronics", 25.00, 5),
        ("Mechanical Keyboard", "Electronics", 75.00, 7),
        ("Running Shoes", "Apparel", 60.00, 10),
        ("Cotton T-Shirt", "Apparel", 15.00, 4),
        ("Coffee Maker", "Home", 45.00, 6)
    ]
    cursor.executemany("INSERT OR IGNORE INTO Products (product_name, category, unit_price, supplier_lead_time_days) VALUES (?, ?, ?, ?)", products)

    # Insert Sample Inventory
    inventory = [
        (1, 150, 30, "2026-10-01"),
        (2, 45, 15, "2026-09-15"),
        (3, 80, 20, "2026-10-10"),
        (4, 300, 50, "2026-10-05"),
        (5, 25, 10, "2026-09-20")
    ]
    cursor.executemany("INSERT OR IGNORE INTO Inventory (product_id, stock_on_hand, safety_stock_threshold, last_restock_date) VALUES (?, ?, ?, ?)", inventory)

    # Insert Sample Transactions (Simulating 60 days of sales)
    base_date = datetime.now() - timedelta(days=60)
    for i in range(1, 200):
        t_date = (base_date + timedelta(days=random.randint(0, 59), hours=random.randint(8, 20))).strftime("%Y-%m-%d %H:%M:%S")
        cust_id = random.randint(1001, 1050)
        prod_id = random.randint(1, 5)
        qty = random.randint(1, 4)
        
        # Get product price to calculate revenue (mocking lookup)
        prices = {1: 25.00, 2: 75.00, 3: 60.00, 4: 15.00, 5: 45.00}
        revenue = round(qty * prices[prod_id], 2)

        cursor.execute('''
            INSERT INTO Transactions (date, customer_id, product_id, quantity_sold, total_revenue)
            VALUES (?, ?, ?, ?, ?)
        ''', (t_date, cust_id, prod_id, qty, revenue))

    conn.commit()
    conn.close()
    print("Database `ecommerce_demand.db` created and populated successfully!")

if __name__ == "__main__":
    setup_ecommerce_database()

Database `ecommerce_demand.db` created and populated successfully!


In [2]:
import sqlite3
import pandas as pd

def build_feature_pipeline(db_name="ecommerce_demand.db"):
    # 1. Connect to SQLite and load transactions joined with product info
    conn = sqlite3.connect(db_name)
    query = """
        SELECT 
            t.transaction_id,
            t.date,
            t.product_id,
            p.product_name,
            p.category,
            t.quantity_sold,
            t.total_revenue
        FROM Transactions t
        JOIN Products p ON t.product_id = p.product_id
    """
    df = pd.read_sql(query, conn)
    conn.close()

    print(f"Loaded {len(df)} transaction rows from database.")

    # 2. Convert date column to datetime objects
    df['date'] = pd.to_datetime(df['date'])

    # 3. Aggregate daily sales by product and date (Daily Time Series)
    daily_sales = df.groupby(
        ['product_id', 'product_name', 'category', pd.Grouper(key='date', freq='D')]
    )['quantity_sold'].sum().reset_index()

    # Sort chronologically by product and date
    daily_sales = daily_sales.sort_values(by=['product_id', 'date'])

    # 4. Extract Calendar/Temporal Features
    daily_sales['year'] = daily_sales['date'].dt.year
    daily_sales['month'] = daily_sales['date'].dt.month
    daily_sales['day_of_week'] = daily_sales['date'].dt.dayofweek
    daily_sales['is_weekend'] = daily_sales['day_of_week'].isin([5, 6]).astype(int)

    # 5. Create Lag Features (What happened yesterday and 7 days ago)
    daily_sales['sales_lag_1'] = daily_sales.groupby('product_id')['quantity_sold'].shift(1)
    daily_sales['sales_lag_7'] = daily_sales.groupby('product_id')['quantity_sold'].shift(7)

    # 6. Create Rolling Window Features (7-day moving average and rolling standard deviation)
    daily_sales['rolling_mean_7'] = daily_sales.groupby('product_id')['quantity_sold'].transform(lambda x: x.rolling(window=7).mean())
    daily_sales['rolling_std_7'] = daily_sales.groupby('product_id')['quantity_sold'].transform(lambda x: x.rolling(window=7).std())

    # Drop rows with NaN values created by lag/rolling shifts
    daily_sales = daily_sales.dropna().reset_index(drop=True)

    print(f"Feature engineering complete. Prepared {len(daily_sales)} rows of model-ready data.")
    
    # Preview the first few rows of engineered features
    print("\nSample Feature-Engineered Data:")
    print(daily_sales[['date', 'product_name', 'quantity_sold', 'sales_lag_1', 'rolling_mean_7']].head())

    # Optional: Save back to a new table or CSV for modeling
    daily_sales.to_csv("engineered_sales_data.csv", index=False)
    print("\nSaved engineered dataset to `engineered_sales_data.csv` successfully!")

    return daily_sales

if __name__ == "__main__":
    build_feature_pipeline()

Loaded 199 transaction rows from database.
Feature engineering complete. Prepared 108 rows of model-ready data.

Sample Feature-Engineered Data:
        date    product_name  quantity_sold  sales_lag_1  rolling_mean_7
0 2026-09-16  Wireless Mouse              7          1.0        3.428571
1 2026-09-17  Wireless Mouse              4          7.0        3.428571
2 2026-09-24  Wireless Mouse              4          4.0        3.285714
3 2026-09-26  Wireless Mouse              5          4.0        3.571429
4 2026-09-28  Wireless Mouse              2          5.0        3.571429

Saved engineered dataset to `engineered_sales_data.csv` successfully!


In [ ]:
import sqlite3
import pandas as pd
import streamlit as st
import plotly.express as px

# Page Configuration
st.set_page_config(
    page_title="E-Commerce Demand & Inventory Dashboard",
    page_icon="📈",
    layout="wide"
)

# Load Data from SQLite
@st.cache_data
def load_data():
    conn = sqlite3.connect("ecommerce_demand.db")
    
    # Load Transactions & Products
    df_trans = pd.read_sql("""
        SELECT t.date, t.customer_id, t.product_id, p.product_name, p.category, t.quantity_sold, t.total_revenue
        FROM Transactions t
        JOIN Products p ON t.product_id = p.product_id
    """, conn)
    
    # Load Inventory Status
    df_inv = pd.read_sql("""
        SELECT i.product_id, p.product_name, p.category, i.stock_on_hand, i.safety_stock_threshold, i.last_restock_date
        FROM Inventory i
        JOIN Products p ON i.product_id = p.product_id
    """, conn)
    
    conn.close()
    df_trans['date'] = pd.to_datetime(df_trans['date'])
    return df_trans, df_inv

df_trans, df_inv = load_data()

# --- SIDEBAR NAVIGATION & FILTERS ---
st.sidebar.header("📊 Control Panel")
selected_category = st.sidebar.selectbox(
    "Filter by Category",
    options=["All"] + list(df_trans['category'].unique())
)

# Filter Data based on sidebar selection
if selected_category != "All":
    filtered_trans = df_trans[df_trans['category'] == selected_category]
    filtered_inv = df_inv[df_inv['category'] == selected_category]
else:
    filtered_trans = df_trans
    filtered_inv = df_inv

# --- MAIN DASHBOARD HEADER ---
st.title(" E-Commerce Demand Forecasting & Inventory Intelligence")
st.markdown("Monitor real-time sales velocity, track stockout risks, and analyze category performance.")

# --- TOP KPI METRICS ROW ---
total_revenue = filtered_trans['total_revenue'].sum()
total_units = filtered_trans['quantity_sold'].sum()
low_stock_items = len(filtered_inv[filtered_inv['stock_on_hand'] <= filtered_inv['safety_stock_threshold']])

col1, col2, col3 = st.columns(3)
col1.metric("Total Revenue", f"${total_revenue:,.2f}")
col2.metric("Total Units Sold", f"{total_units:,}")
col3.metric("Stockout Alerts", f"{low_stock_items} Items", delta_color="inverse" if low_stock_items > 0 else "normal")

st.markdown("---")

# --- TABS FOR ORGANIZED LAYOUT ---
tab1, tab2, tab3 = st.tabs(["📈 Sales & Demand Trends", "⚠️ Inventory & Stock Risk", "📋 Raw Data Explorer"])

with tab1:
    st.subheader("Daily Sales Velocity by Product")
    
    # Aggregate daily sales for plotting
    daily_trend = filtered_trans.groupby([pd.Grouper(key='date', freq='D'), 'product_name'])['quantity_sold'].sum().reset_index()
    
    fig_trend = px.line(
        daily_trend, 
        x='date', 
        y='quantity_sold', 
        color='product_name',
        labels={'quantity_sold': 'Units Sold', 'date': 'Date', 'product_name': 'Product'},
        template='plotly_white'
    )
    st.plotly_chart(fig_trend, use_container_width=True)

with tab2:
    st.subheader("Inventory Health & Safety Thresholds")
    st.markdown("Products where **Stock on Hand** is approaching or below the **Safety Threshold** require reordering.")
    
    # Highlight low stock items
    fig_inv = px.bar(
        filtered_inv,
        x='product_name',
        y=['stock_on_hand', 'safety_stock_threshold'],
        barmode='group',
        labels={'value': 'Quantity', 'product_name': 'Product Name', 'variable': 'Metric'},
        template='plotly_white'
    )
    st.plotly_chart(fig_inv, use_container_width=True)
    
    # Warning table for low stock
    at_risk = filtered_inv[filtered_inv['stock_on_hand'] <= filtered_inv['safety_stock_threshold']]
    if not at_risk.empty:
        st.warning(f"⚠️ Action Required: {len(at_risk)} product(s) are at or below safety stock limits!")
        st.dataframe(at_risk, use_container_width=True)
    else:
        st.success("✅ All inventories are currently above safety thresholds.")

with tab3:
    st.subheader("Transactional Records Explorer")
    st.dataframe(filtered_trans, use_container_width=True)

2026-11-03 10:00:27.556 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-03 10:00:27.568 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-03 10:00:27.575 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-03 10:00:27.577 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-03 10:00:27.818 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-03 10:00:29.289 
  command:

    streamlit run C:\Users\brian\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312